# Notebook 01-UCI - Feature engineering for the second dataset (UCI 697)

This builds the second-dataset arm of the cross-dataset audit. The trust-equity audit protocol is
applied identically to OULAD; only the feature engineering differs, because this dataset has no
clickstream to window. Instead of weekly cutoffs it has enrollment-time and semester snapshots.

**Dataset.** Predict Students' Dropout and Academic Success (Realinho, Vieira Martins, Machado, and
Baptista, 2021), UCI Machine Learning Repository id 697, CC-BY 4.0. 4,424 students, three-class
outcome (Dropout / Enrolled / Graduate).

**Label.**
* Primary: Dropout = 1, Graduate = 0, Enrolled rows removed (unresolved outcome).
* Robustness (fold): Dropout = 1, Graduate and Enrolled = 0 (all rows kept). This mirrors the
  Fail-only versus Fail-or-Withdrawn robustness on OULAD.

**Snapshots (cross-granularity early warning).**
* T0 enrollment-only: demographics, admission, prior qualification, course, macroeconomic context.
* T1 enrollment + semester 1: T0 plus first-semester curricular-unit features.
* T2 enrollment + semester 1 + semester 2: appendix only, reported as a late-snapshot upper bound /
  leakage-sensitivity check, because second-semester fields are near-proxies for having already left.

**Audit axes (sensitive-attribute transfer).** Protected and audit-axis attributes are excluded from
the model inputs and kept only for the audit, mirroring OULAD.
* Disability-analogue: educational special needs (binary).
* Socioeconomic-support proxy: scholarship holder (binary, direction-agnostic - reported with its base
  rate, since scholarship may be means-tested or merit-based).
* Secondary financial-stress axes: debtor, tuition fees up to date, and a composite
  financial_vulnerability = debtor OR not-tuition-up-to-date (outcome-proximal, robustness only).
* Also retained: gender and an age group (tertile-extreme) to mirror OULAD's gender and age axes.

**Outputs (in results/processed/):** `uci_model_ready_T0`, `_T1`, `_T2`; `uci_features.json`
(feature lists per snapshot); `uci_label_summary.csv` (subgroup sizes and base rates).

## 0. Setup

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ucimlrepo'])

from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')
PROC = ROOT / 'results' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
SEED = 42

Mounted at /content/drive


In [2]:
import re, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Fetch UCI 697

In [3]:
from ucimlrepo import fetch_ucirepo
ds = fetch_ucirepo(id=697)
df_raw = ds.data.features.copy()
df_raw['Target'] = ds.data.targets.iloc[:, 0].values
print('fetched', df_raw.shape, 'rows x cols')
print('raw target values:', df_raw['Target'].value_counts().to_dict())

fetched (4424, 37) rows x cols
raw target values: {'Graduate': 2209, 'Dropout': 1421, 'Enrolled': 794}


## 2. Normalise columns and identify roles

In [4]:
def norm(c): return re.sub(r'[^a-z0-9]+', '_', str(c).strip().lower()).strip('_')
df = df_raw.copy()
df.columns = [norm(c) for c in df.columns]
cols = list(df.columns)

def find(*subs):
    for c in cols:
        if all(s in c for s in subs):
            return c
    return None

col_special = find('special', 'needs')
col_scholar = find('scholarship')
col_debtor  = find('debtor')
col_tuition = find('tuition')
col_gender  = find('gender')
col_age     = find('age', 'enroll') or find('age')
col_intl    = find('international')
col_displ   = find('displaced')
col_marital = find('marital')
col_nation  = find('nacional') or find('national')
col_target  = 'target' if 'target' in cols else find('target')

SEM1 = [c for c in cols if '1st_sem' in c]
SEM2 = [c for c in cols if '2nd_sem' in c]

print('special needs col :', col_special)
print('scholarship col   :', col_scholar)
print('debtor / tuition  :', col_debtor, '/', col_tuition)
print('gender / age      :', col_gender, '/', col_age)
print('target col        :', col_target)
print('sem1 features (%d):' % len(SEM1), SEM1)
print('sem2 features (%d):' % len(SEM2), SEM2)
assert all(x is not None for x in [col_special, col_scholar, col_debtor, col_tuition,
                                   col_gender, col_age, col_target]), 'a key column was not found - check normalised names above'

special needs col : educational_special_needs
scholarship col   : scholarship_holder
debtor / tuition  : debtor / tuition_fees_up_to_date
gender / age      : gender / age_at_enrollment
target col        : target
sem1 features (6): ['curricular_units_1st_sem_credited', 'curricular_units_1st_sem_enrolled', 'curricular_units_1st_sem_evaluations', 'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade', 'curricular_units_1st_sem_without_evaluations']
sem2 features (6): ['curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled', 'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations']


## 3. Audit axes and labels

In [5]:
out = pd.DataFrame(index=df.index)
out['student_id'] = np.arange(len(df))

def yn(series, positive=1): return np.where(pd.to_numeric(series, errors='coerce') == positive, 'Y', 'N')
out['disability']         = yn(df[col_special])            # educational special needs
out['scholarship']        = yn(df[col_scholar])            # SES-support proxy
out['debtor']             = yn(df[col_debtor])
out['tuition_up_to_date'] = yn(df[col_tuition])
out['fin_vulnerable']     = np.where((pd.to_numeric(df[col_debtor], errors='coerce') == 1) |
                                     (pd.to_numeric(df[col_tuition], errors='coerce') == 0), 'Y', 'N')
out['gender']             = np.where(pd.to_numeric(df[col_gender], errors='coerce') == 1, 'M', 'F')

age = pd.to_numeric(df[col_age], errors='coerce')
q1, q2 = age.quantile([1/3, 2/3]).values
out['age_group'] = np.where(age <= q1, 'young', np.where(age >= q2, 'older', 'mid'))

t = df[col_target].astype(str).str.strip().str.lower()
out['target']        = t
out['enrolled_flag'] = (t == 'enrolled').astype(int)
out['at_risk']       = np.where(t == 'dropout', 1.0, np.where(t == 'graduate', 0.0, np.nan))  # enrolled = NaN
out['at_risk_fold']  = (t == 'dropout').astype(int)                                            # graduate + enrolled = 0
print('primary positive rate (drop Enrolled):',
      round(out.loc[out.enrolled_flag == 0, 'at_risk'].mean(), 4))
print('fold positive rate (Enrolled as negative):', round(out['at_risk_fold'].mean(), 4))

primary positive rate (drop Enrolled): 0.3915
fold positive rate (Enrolled as negative): 0.3212


## 4. Feature selection and leakage guard

In [6]:
# attributes excluded from model inputs (protected / audit axes / identity), mirroring OULAD
EXCLUDE = {c for c in [col_special, col_scholar, col_debtor, col_tuition, col_gender,
                       col_age, col_intl, col_displ, col_marital, col_nation, col_target] if c}

base_feats = [c for c in cols if c not in EXCLUDE and c not in SEM1 and c not in SEM2]
# nominal categoricals to one-hot: course, application mode, qualifications, occupations (not grades)
NOMINAL = [c for c in base_feats
           if any(k in c for k in ['application_mode', 'course', 'qualification', 'occupation'])
           and 'grade' not in c]
NUMERIC = [c for c in base_feats if c not in NOMINAL]

print('T0 base features:', len(base_feats), '| nominal (one-hot):', len(NOMINAL), '| numeric:', len(NUMERIC))
print('nominal:', NOMINAL)
print('numeric:', NUMERIC)

# leakage guards
assert not (set(SEM1) & set(base_feats)), 'sem1 leaked into T0'
assert not (set(SEM2) & set(base_feats)), 'sem2 leaked into T0'
assert not (set(SEM2) & set(SEM1)), 'sem2 leaked into sem1'
print('leakage guard passed: T0 has no semester features; T1 adds only sem1; T2 adds sem2')

T0 base features: 14 | nominal (one-hot): 7 | numeric: 7
nominal: ['application_mode', 'course', 'previous_qualification', 'mother_s_qualification', 'father_s_qualification', 'mother_s_occupation', 'father_s_occupation']
numeric: ['application_order', 'daytime_evening_attendance', 'previous_qualification_grade', 'admission_grade', 'unemployment_rate', 'inflation_rate', 'gdp']
leakage guard passed: T0 has no semester features; T1 adds only sem1; T2 adds sem2


## 5. Encode, split, and assemble snapshot tables

In [7]:
# one-hot the nominal features (encoding on the full frame is not leakage)
nom = df[NOMINAL].apply(lambda s: pd.to_numeric(s, errors='coerce')).astype('Int64').astype(str)
dummies = pd.get_dummies(nom, prefix=NOMINAL, dummy_na=False).astype(int)
num = df[NUMERIC].apply(lambda s: pd.to_numeric(s, errors='coerce'))
sem1 = df[SEM1].apply(lambda s: pd.to_numeric(s, errors='coerce')) if SEM1 else pd.DataFrame(index=df.index)
sem2 = df[SEM2].apply(lambda s: pd.to_numeric(s, errors='coerce')) if SEM2 else pd.DataFrame(index=df.index)

T0_FEATURES = list(num.columns) + list(dummies.columns)
T1_FEATURES = T0_FEATURES + list(sem1.columns)
T2_FEATURES = T1_FEATURES + list(sem2.columns)
Xall = pd.concat([num, dummies, sem1, sem2], axis=1)

# stratified 60/20/20 split on the 3-class target (keeps both labels balanced)
idx = np.arange(len(out))
tr, tmp = train_test_split(idx, test_size=0.4, stratify=out['target'], random_state=SEED)
ca, te = train_test_split(tmp, test_size=0.5, stratify=out['target'].iloc[tmp], random_state=SEED)
split = np.array(['train'] * len(out), dtype=object); split[ca] = 'calib'; split[te] = 'test'
out['split'] = split

META = ['student_id','disability','scholarship','debtor','tuition_up_to_date','fin_vulnerable',
        'gender','age_group','target','enrolled_flag','at_risk','at_risk_fold','split']

def save_ready(feats, name):
    tbl = pd.concat([out[META].reset_index(drop=True), Xall[feats].reset_index(drop=True)], axis=1)
    try: tbl.to_parquet(PROC / f'{name}.parquet', index=False)
    except Exception: tbl.to_csv(PROC / f'{name}.csv', index=False)
    return tbl

save_ready(T0_FEATURES, 'uci_model_ready_T0')
save_ready(T1_FEATURES, 'uci_model_ready_T1')
save_ready(T2_FEATURES, 'uci_model_ready_T2')
with open(ROOT / 'results' / 'uci_features.json', 'w') as f:
    json.dump({'T0': T0_FEATURES, 'T1': T1_FEATURES, 'T2': T2_FEATURES}, f, indent=1)
print('saved uci_model_ready_T0/T1/T2 and uci_features.json')
print('feature counts  T0:', len(T0_FEATURES), '| T1:', len(T1_FEATURES), '| T2:', len(T2_FEATURES))
print('split sizes:', {k:int((split==k).sum()) for k in ['train','calib','test']})

saved uci_model_ready_T0/T1/T2 and uci_features.json
feature counts  T0: 200 | T1: 206 | T2: 212
split sizes: {'train': 2654, 'calib': 885, 'test': 885}


## 6. Subgroup sizes and base rates (read this before modelling)

In [8]:
res = out[out['enrolled_flag'] == 0]   # resolved rows = primary-label population
rows = []
for axis in ['scholarship','disability','age_group','gender','debtor','fin_vulnerable']:
    g = res.groupby(axis)['at_risk'].agg(['size','mean'])
    for level, r in g.iterrows():
        rows.append({'axis': axis, 'group': level, 'n': int(r['size']), 'dropout_rate': round(r['mean'], 4)})
summary = pd.DataFrame(rows)
summary.to_csv(ROOT / 'results' / 'uci_label_summary.csv', index=False)
print(summary.to_string(index=False))
print()
print('NOTE: check the scholarship rows above. If scholarship holders have a LOWER dropout rate than')
print('non-holders, the SES proxy runs opposite to OULAD deprivation, and the transfer test is')
print('direction-agnostic (the audit machinery transfers; the group-risk direction is dataset-specific).')
print('Also check the special-needs (disability) n: if small, treat any null there as underpowered.')

          axis group    n  dropout_rate
   scholarship     N 2661        0.4837
   scholarship     Y  969        0.1383
    disability     N 3590        0.3911
    disability     Y   40        0.4250
     age_group   mid  711        0.3179
     age_group older 1298        0.6055
     age_group young 1621        0.2523
        gender     F 2381        0.3024
        gender     M 1249        0.5612
        debtor     N 3217        0.3447
        debtor     Y  413        0.7554
fin_vulnerable     N 2957        0.2932
fin_vulnerable     Y  673        0.8232

NOTE: check the scholarship rows above. If scholarship holders have a LOWER dropout rate than
non-holders, the SES proxy runs opposite to OULAD deprivation, and the transfer test is
direction-agnostic (the audit machinery transfers; the group-risk direction is dataset-specific).
Also check the special-needs (disability) n: if small, treat any null there as underpowered.


## What was built, and what to check

Three snapshot tables (`uci_model_ready_T0/T1/T2`), each with the audit-axis columns, both labels
(`at_risk` primary with Enrolled as NaN, `at_risk_fold` robustness), a stratified split, and the
snapshot's model-input features; plus the feature lists and a subgroup base-rate summary.

Before I build the modelling and audit notebooks, paste me the printed output of sections 2, 5, and 6
so I can confirm: the real column names mapped correctly (section 2), the feature counts and split are
sensible (section 5), and crucially the subgroup base rates (section 6) - especially the direction of
the scholarship signal and the size of the special-needs group, since those shape how the
sensitive-attribute transfer is honestly framed.